# SESMG Scenario Runner — the method

This notebook documents the **core loop** for running building energy-system optimizations
through SESMG (the `esmp` package) in batch. It is written so the pattern is easy to see and
**adapt to the other scenarios**.

The method is scenario-independent. For each building combination it:

1. **reads** the combination (building type, city, heat demand, refurbishment state)
2. **builds** a model JSON from a scenario template, patched with that building's values
3. **runs** `esmp optimize` on it
4. **collects** the results (costs, capacities, hourly profiles)
5. **writes** one row to a per-scenario table

Below, each step is a cell you can run and inspect. The **gas-only** scenario is used as the
worked example (36/36 buildings solve). At the end, a short section explains exactly what to
change for the electricity, PV, and heat-pump scenarios — the loop itself does not change.

## 0. Setup

Paths and constants. Only these change between machines. `TEMPLATE` is the one thing that
changes between *scenarios* — it points at a known-good example in `tests/scenarios/`.

In [9]:
import os
os.environ["MPLBACKEND"] = "Agg"

import matplotlib
matplotlib.use("Agg")

import json, shutil, subprocess, csv
from pathlib import Path

# --- machine paths -------------------------------------------------------
ESMP_ROOT   = Path(r"C:\Users\zaito\esmp")
COMBOS_FILE = Path(r"C:\Users\zaito\futurebeeing\combinations_heat_only.txt")
WORK_DIR    = ESMP_ROOT / "work_folder"
OUT_CSV     = Path(r"C:\Users\zaito\futurebeeing\scenario_gas_only_results.csv")
SOLVER      = "cbc"          # cbc needs no licence; swap to "gurobi" if licensed

# --- the scenario template (THIS is what changes per scenario) -----------
TEMPLATE    = ESMP_ROOT / "tests/scenarios/Waerme_Gasheizung_Gasleitung"   # gas only

# --- NL constants (from the official RVO list) ---------------------------
GAS_EMIS_G_PER_KWH    = 202.0     # combustion basis, co2emissiefactoren.nl
GAS_PRICE_EUR_PER_KWH = 0.10      # PLACEHOLDER - awaiting confirmation

LIMIT = 1     # run 1 combination while testing; set to None for all 36
print("configured. template:", TEMPLATE.name)

configured. template: Waerme_Gasheizung_Gasleitung


## 1. Read the combinations

Each line of the combination file is a building to optimize. The file name encodes its schema
in the header, e.g. `size_class_nearest_city_heat_cluster_refurbishment_state`.

**Parsing note:** the delimiter is `_`, and the *field names* contain underscores too — but the
*values* are underscore-free by design (that is why the city is "DeBilt", not "De Bilt"). So a
plain split on `_` gives exactly the right number of fields. Each scenario's combination file
has a different schema, so the parser is the one part that changes with the input file.

In [10]:
def read_combos(path):
    lines = Path(path).read_text().strip().splitlines()
    combos = []
    for ln in lines[1:]:                       # skip the header line
        parts = ln.strip().split("_")
        if len(parts) != 4:                    # gas-only schema has 4 fields
            print("  ! skipping malformed line:", ln); continue
        size_class, city, heat_cluster, refurb = parts
        combos.append({
            "size_class":   size_class,
            "city":         city,
            "heat_cluster": float(heat_cluster),   # annual heat demand, kWh/yr
            "refurb_state": int(refurb),           # 1 / 2 / 3
        })
    return combos

combos = read_combos(COMBOS_FILE)
print(f"read {len(combos)} combinations. first one:")
print(" ", combos[0])

read 36 combinations. first one:
  {'size_class': 'MFH', 'city': 'DeBilt', 'heat_cluster': 8417.59, 'refurb_state': 2}


## 2. Build the model for one combination

Start from the template's three JSONs (`model.json`, `timeseries.json`, `weather.json`) and
**patch** the parts specific to this building. For gas-only, two things are patched:

- **heat demand** — the heat sink's `annualDemand` is set to this building's kWh. `esmp` builds
  the hourly curve from it.
- **capacity ceilings** — the converter and gas-bus limits are raised so large buildings are not
  artificially blocked. (Per Philippe: set them high, the solver finds the optimum either way.
  Without this, large apartment blocks return "infeasible".)

**This is the cell that changes most per scenario** — a PV scenario patches roof area, a
heat-pump scenario patches the flow temperature, etc. What is patched depends on the template's
components.

In [11]:
def prepare_work_folder(combo):
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)

    # timeseries + weather define the hourly SHAPE - copied unchanged
    shutil.copy(TEMPLATE / "timeseries.json", WORK_DIR / "timeseries.json")
    shutil.copy(TEMPLATE / "weather.json",   WORK_DIR / "weather.json")

    model = json.loads((TEMPLATE / "model.json").read_text())

    patched = False
    for comp in model["components"]:
        # (a) scale annual heat demand to this cluster
        if comp.get("category") == "sink" and comp.get("sector") == "heat":
            comp["annualDemand"] = combo["heat_cluster"]
            patched = True
        # (b) raise capacity ceilings so large buildings are not blocked
        if comp.get("category") == "converter":
            comp["maxInvestmentCapacity"] = 1_000_000
        if comp.get("category") == "bus" and "shortage" in comp:
            comp["shortage"]["capacity"] = 1_000_000
    if not patched:
        raise RuntimeError("heat sink not found in model.json - check template structure")

    (WORK_DIR / "model.json").write_text(json.dumps(model, indent=2))

# demo on the first combination
prepare_work_folder(combos[0])
print("work folder ready for", combos[0]["size_class"], combos[0]["heat_cluster"], "kWh")

work folder ready for MFH 8417.59 kWh


## 3. Run the optimizer

Call `esmp optimize` on the work folder. `esmp` reads the three JSONs, solves the least-cost
model, and writes result CSVs back into the same folder. This cell is **identical for every
scenario** — it does not change.

In [12]:
def run_optimize():
    cmd = f'uv run esmp optimize --path "{WORK_DIR}" --solver {SOLVER}'
    r = subprocess.run(cmd, shell=True, cwd=str(ESMP_ROOT),
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError("esmp optimize failed")
    return r

run_optimize()
print("optimization done. result files in", WORK_DIR.name, ":")
for f in ["summary.csv", "components.csv", "results.csv"]:
    print("  -", f, "exists:", (WORK_DIR / f).exists())

optimization done. result files in work_folder :
  - summary.csv exists: True
  - components.csv exists: True
  - results.csv exists: True


## 4. Collect the results

Read the values Philippe listed out of the three result files:

- **`summary.csv`** — system totals (periodical cost, energy demand)
- **`components.csv`** — per-component values (capacity, production, O&M, investment)
- **`results.csv`** — the hourly flows; each column is one flow, one row per hour. Summing a
  column gives an annual total; keeping the column gives a load profile.

Emissions and import cost are **computed** here from the import quantity times the NL constants
(they are not solver outputs). **This cell grows per scenario** — a PV scenario also reads the
PV-generation and export flows, etc. — but the pattern (read summary + components + results) is
the same.

In [13]:
def collect_results(combo):
    row = dict(combo)

    # summary.csv - system scalars
    with open(WORK_DIR / "summary.csv") as f:
        s = list(csv.DictReader(f))[0]
    row["cost_periodical"]      = float(s["Total Periodical Costs"])
    row["total_variable_costs"] = float(s["Total Variable Costs"])
    row["total_energy_demand"]  = float(s["Total Energy Demand"])

    # components.csv - per-component values
    with open(WORK_DIR / "components.csv") as f:
        comps = list(csv.DictReader(f))
    conv = next((c for c in comps if c["type"] == "converter"), None)
    if conv:
        row["cap_invest"]  = float(conv["capacity/kW"])
        row["production"]  = float(conv["output 1/kWh"])
        row["costs_om"]    = float(conv["variable costs/CU"])
        row["cost_invest"] = float(conv["investment/kW"])
    imp = sum(float(c["output 1/kWh"]) for c in comps if c["type"] == "source")
    row["energy_import"] = imp

    # computed (not solver outputs)
    row["import_cost"] = imp * GAS_PRICE_EUR_PER_KWH
    row["emissions"]   = imp * GAS_EMIS_G_PER_KWH

    # results.csv - hourly profiles (kept as lists)
    with open(WORK_DIR / "results.csv") as f:
        rd = csv.DictReader(f)
        cols = {c: [] for c in rd.fieldnames if c != "date"}
        for r in rd:
            for c in cols: cols[c].append(float(r[c]))
    heat_col = next((c for c in cols if "heat_sink_input1" in c), None)
    row["heat_demand_profile"] = cols.get(heat_col, [])

    return row

demo = collect_results(combos[0])
print("collected values for", combos[0]["size_class"], ":")
for k in ["cost_periodical","cap_invest","production","energy_import","emissions"]:
    print(f"  {k:<18}{demo[k]:,.1f}")
print(f"  heat profile: {len(demo['heat_demand_profile'])} hourly values")

collected values for MFH :
  cost_periodical   200.4
  cap_invest        4.0
  production        8,389.2
  energy_import     8,389.2
  emissions         1,694,620.4
  heat profile: 2182 hourly values


## 5. The full loop

Put the four steps together and run over every combination. Failures are caught per building so
one bad case does not stop the batch; the rest still write. Profiles are stored as JSON strings
in the CSV cells. **Set `LIMIT = None` above to run all combinations.**

In [14]:
def run_all():
    todo = combos if LIMIT is None else combos[:LIMIT]
    print(f"running {len(todo)} combination(s) with solver={SOLVER}")
    rows = []
    for i, combo in enumerate(todo, 1):
        cid = f'{combo["size_class"]}_{combo["city"]}_{combo["heat_cluster"]}_{combo["refurb_state"]}'
        print(f"[{i}/{len(todo)}] {cid}")
        try:
            prepare_work_folder(combo)
            run_optimize()
            rows.append(collect_results(combo))
        except Exception as e:
            print("  FAILED:", e)
    if not rows:
        print("no successful runs"); return
    keys = list(rows[0].keys())
    with open(OUT_CSV, "w", newline="") as f:
        w = csv.writer(f); w.writerow(keys)
        for r in rows:
            w.writerow([json.dumps(v) if isinstance(v, list) else v
                        for v in (r[k] for k in keys)])
    print(f"wrote {len(rows)} row(s) -> {OUT_CSV}")

run_all()

running 1 combination(s) with solver=cbc
[1/1] MFH_DeBilt_8417.59_2
wrote 1 row(s) -> C:\Users\zaito\futurebeeing\scenario_gas_only_results.csv


In [22]:
m = json.loads((TEMPLATE / "model.json").read_text())
print(json.dumps(m["energysystem"], indent=2))

{
  "startDate": "2012-01-01T00:00:00",
  "endDate": "2012-12-30T23:00:00",
  "timezone": "Europe/Berlin",
  "temporalResolution": "h",
  "periods": 8760,
  "constraintCostLimit": null,
  "minFinalEnergyReduction": 0
}


## 6. How to adapt this to the other scenarios

**The loop does not change.** Only three things change per scenario, and each maps to one cell
above:

| Scenario | Template (cell 0) | Patch (cell 2) | Collect (cell 4) |
|---|---|---|---|
| **Gas only** *(done)* | `Waerme_Gasheizung_Gasleitung` | heat demand | heat profile, gas import |
| **Electricity only** | `Strom_...` (elec template) | electricity demand | elec demand profile, grid import |
| **PV only** | `Strom_PV` | roof area (east/west/south) | PV generation, export, import profiles |
| **Heat pump + PV** | `Strom_Waerme_PV_Waermepumpe` | heat demand + roof + flow temp (refurb→60/50/35 °C) | heat + elec + PV profiles, COP |

**To add a scenario:**

1. **Point `TEMPLATE`** at the matching example folder in `tests/scenarios/`.
2. **Open that template's `model.json`** and see which components it has and what needs scaling
   per building (e.g. PV needs the roof-area fields; the heat pump needs `temperatureHigh`).
   Update the patch cell accordingly.
3. **Run one combination**, look at the result CSVs, and add the new columns to the collect cell
   (PV adds generation/export/import flows, etc.).
4. **Point `COMBOS_FILE`** at that scenario's combination file (each has its own schema — update
   the parser in cell 1 to match its fields).

**Notes carried from the gas-only work:**
- For **gas-only, refurbishment state does not change the result** (fixed boiler efficiency), so
  same-demand/different-refurb rows are identical. For the **heat-pump** scenario it *will*
  matter (COP depends on flow temperature).
- **Emissions**: gas uses 202 gCO2/kWh (combustion). Electricity uses the NL grid factor
  (~268 gCO2/kWh, RVO 2025) for the scenarios with electricity import.
- **Gas price** is still a placeholder pending confirmation.